In [ ]:
# Creating SparkSession
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName("DAG INFO")
    .master("local[*]")
    .getOrCreate()
)

In [ ]:
spark


In [ ]:
# Disable or Deactive AQE(adaptive Query Execution)
spark.conf.set("spark.sql.adaptive.enabled",False)
spark.conf.set("spark.sql.adaptive.coalescepartitions.enabled",False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",-1)

In [ ]:
df_1=spark.range(4,200,2)
df_2=spark.range(2,200,4)

In [ ]:
df_3=df_1.repartition(5)
df_4=df_2.repartition(7)

In [ ]:
df_1.rdd.getNumPartitions()

8

In [ ]:
df_joined=df_3.join(df_4,on="id")

In [ ]:
df_sum=df_joined.selectExpr("sum(id) as total_sum")

In [ ]:
df_sum.show()

+---------+
|total_sum|
+---------+
|     4998|
+---------+



In [ ]:
df_sum.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[sum(id#0L)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [id=#204]
      +- HashAggregate(keys=[], functions=[partial_sum(id#0L)])
         +- Project [id#0L]
            +- BroadcastHashJoin [id#0L], [id#2L], Inner, BuildRight, false
               :- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [id=#191]
               :  +- Range (4, 200, step=2, splits=8)
               +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [id=#199]
                  +- Exchange RoundRobinPartitioning(7), REPARTITION_BY_NUM, [id=#193]
                     +- Range (2, 200, step=4, splits=8)




In [ ]:
df_union=df_sum.union(df_4)
